# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² dataset using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset source is defined via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a single object, not a dictionary

print(f"Dataset title: {getattr(metadata, 'name', '<missing>')}")
print("\nDescription:")
print(getattr(metadata, 'description', '<missing>'))

## 2. Data Overview
Review available record sets, their fields, columns, and associated `@id` values. All exploration uses Croissant schema IDs for clarity and reproducibility.

In [ ]:
# Discover available record sets
record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f"[{i}] @id: {getattr(rs, '@id', '<missing>')}")
    print(f"    Name: {getattr(rs, 'name', '<unnamed>')}")
    print(f"    Description: {getattr(rs, 'description', '<missing>')}")
    # List the field @ids
    if hasattr(rs, 'fields'):
        print(f"    Fields:")
        for f in rs.fields:
            print(f"        @id: {getattr(f, '@id', '<missing>')}, name: {getattr(f, 'name', '<missing>')}")
    print("")

# Show example records for first record set, if available
if record_sets:
    rs_id = getattr(record_sets[0], '@id')
    print(f"Showing up to 2 records from record set @id: {rs_id}")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(f"Record {i+1}:")
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Extract all record sets' data into DataFrames using their `@id`s, so we can perform further analysis. Note that all references are made by `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [getattr(rs, '@id') for rs in record_sets]
dataframes = {}
# Also store field mappings for each record set for later use
fields_by_record_set = {}

for rs in record_sets:
    rs_id = getattr(rs, '@id')
    # Get all records (each is a dictionary of field values)
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    if hasattr(rs, 'fields'):
        fields_by_record_set[rs_id] = [(getattr(f, '@id'), getattr(f, 'name', '')) for f in rs.fields]
    else:
        fields_by_record_set[rs_id] = []

# Display the columns of the first record set DataFrame
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in first record set (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard preprocessing: select fields, filter/clean, normalize, and group by attributes.

_All attributes below are referenced using their Croissant `@id` fields._

In [ ]:
# Pick the first record set for EDA
"""
Edit these variables based on overview above if needed. For example, set the numeric_field_id
to the actual @id of a numeric field in the selected record set. If you are unsure, display the columns first.
"""
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    # Try to pick a numeric field automatically
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Fallback: try to coerce columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notnull().sum() > 0:
                    numeric_field_id = col
                    break
            except:
                pass

    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First rows with normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to select a groupable field (categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < df.shape[0] // 2:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found in the first record set. Please inspect the columns above.")

## 5. Visualization
Visualize distributions or relationships for fields using the DataFrame. You may need to edit field names below if your record set uses different ones.

In [ ]:
# Example: histogram and boxplot for numeric field, barplot for group field
if record_set_ids and numeric_field_id:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[numeric_field_id].hist(ax=axes[0], bins=15, color='skyblue')
    axes[0].set_title(f"Histogram of {numeric_field_id}")
    axes[0].set_xlabel(numeric_field_id)
    axes[0].set_ylabel('Count')

    df.boxplot(column=numeric_field_id, ax=axes[1])
    axes[1].set_title(f"Boxplot of {numeric_field_id}")

    plt.tight_layout()
    plt.show()

    # If group_field, plot grouped mean
    if 'group_field' in locals() and group_field:
        grouped_df = df.groupby(group_field)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field_id], color='teal')
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-structured dataset using `mlcroissant`:

- **Dataset introspection**: We listed all record sets and their fields by Croissant `@id`.
- **Data extraction**: Loaded all record sets as DataFrames by `@id`.
- **EDA**: Performed filtering, normalization, and basic groupby-aggregation using Croissant IDs for all fields.
- **Visualization**: Plotted histograms, boxplots, and grouped bar charts.

To perform deeper analyses (e.g., hypothesis tests, modeling), continue with standard pandas/numpy/scikit-learn workflows, always referencing fields by their `@id` for reproducibility.

**Note**: If the data appears empty, check that your record set and field IDs match the structure in the schema or print more records for inspection.